In [1]:

from __future__ import annotations

from pathlib import Path
from typing import List
import argparse
import random
import sys

from pff import config_dir, reports_dir
from pff.config_classes.node_pk_config import NodePKConfig
from pff.datasets.aicme_datasets import (
    AICMECompartmentsDataBatch,
    AICMECompartmentsDataModule,
)
from pff.models.amortized_inference.aicme import AICMEPK
from pff.utils.plots.databatch_plot import (
    plot_list_list_study_json,
    plot_list_aicme_databatch,
    plot_study_json
)
from pff.data_empirical.json_schema import (
    StudyJSON,
    canonicalize_study,
    prediction_stats,
)


In [2]:
"""Sample predictions on synthetic and empirical batches, print shapes, and plot.

The utility first draws a batch list from a synthetic ``AICME`` data module
to compare tensor dimensions against an empirical ``StudyJSON`` dataset. For
both sources it samples stochastic predictions using
``model.sample_individual_prediction`` and reports the shapes of the context,
target, and predicted trajectories before rendering the empirical predictions
as a grid of ``StudyJSON`` records.

The shapes follow the project's conventions:
- Each batch has ``target_obs`` and ``context_obs`` tensors shaped
  ``[B, I, T, 1]``.
- ``model.sample_individual_prediction`` returns ``prediction_sample`` and
  ``prediction_time`` shaped ``[S, B, It, Tr, 1]``.
- Empirical predictions are converted to ``StudyJSON`` via
  :func:`prediction_to_study_jsons` and rendered as a grid.
"""

from __future__ import annotations

from pathlib import Path
from typing import List
import argparse
import sys

from torchtyping import TensorType as TT

from pff import config_dir, data_dir, reports_dir
from pff.config_classes.node_pk_config import NodePKConfig
from pff.data_empirical import (
    load_empirical_json_batches,
    load_empirical_hf_batches_as_dm,
    prediction_to_study_jsons,
)
from pff.datasets.aicme_datasets import (
    AICMECompartmentsDataBatch,
    AICMECompartmentsDataModule,
)
from pff.models.amortized_inference.aicme import AICMEPK
from pff.data_empirical.json_schema import StudyJSON
from pff.utils.plots.databatch_plot import (
    plot_list_list_study_json,
    plot_list_aicme_databatch
)

In [3]:
from pathlib import Path
from dataclasses import dataclass

@dataclass
class NotebookConfig:
    yaml: Path
    split: str
    json: Path
    out: Path
    samples: int

    @classmethod
    def default(cls, config_dir: Path, data_dir: Path, reports_dir: Path):
        default_yaml = Path(config_dir) / "experiment_configs" / "node-pk" / "base-homogeneous.yaml"
        default_json = Path(data_dir) / "preprocessed" / "lenuzza_2016.json"
        default_out = Path(reports_dir) / "empirical_prediction_plot.png"

        return cls(
            yaml=default_yaml,
            split="train",
            json=default_json,
            out=default_out,
            samples=8,
        )
    
# Get defaults
args = NotebookConfig.default(config_dir, data_dir, reports_dir)

In [4]:
# 1) Load configuration ---------------------------------------------------
cfg: NodePKConfig = NodePKConfig.from_yaml(str(args.yaml))

# 2) Build synthetic data module and model -------------------------------
dm = AICMECompartmentsDataModule(cfg)
dm.prepare_data()
dm.setup()

model = AICMEPK(cfg)
model.eval()

# 3) Pull a batch list from requested split and inspect shapes ----------
if args.split == "train":
    loader = dm.train_dataloader()
elif args.split == "val":
    loader = dm.val_dataloader()
else:
    loader = dm.test_dataloader()

synthetic_batch_list: List[AICMECompartmentsDataBatch] = next(iter(loader))
synthetic_batch = synthetic_batch_list[0]

In [5]:
synthetic_batch.context_obs.shape

torch.Size([16, 9, 11, 1])

In [9]:
model(synthetic_batch_list,return_forward_report=True)

IndexError: list index out of range